# Knowledge tests


In [1]:
import sys
sys.path.append("..")

## M3 - B+ implementation

### Simple work

In [2]:
papel_em_branco = bytearray(4096)

papel_em_branco[0] = 1 #significa que é folha

papel_em_branco[1] = 0 
papel_em_branco[2] = 0 

print("envelope criado e configurado")

envelope criado e configurado


In [3]:
print("O nó é folha?", papel_em_branco[0] == 1)
print("Quantas chaves tem?", papel_em_branco[1] * 256 + papel_em_branco[2])

O nó é folha? True
Quantas chaves tem? 0


In [4]:
chave_bytes = (42).to_bytes(4, byteorder='big')
papel_em_branco[7:11] = chave_bytes

ptr_bytes = (100).to_bytes(4, byteorder='big')
papel_em_branco[11:15] = ptr_bytes

# atualiza a quantidade de chaves

papel_em_branco[1] = 0
papel_em_branco[2] = 1

print("Chave e ponteiro gravados")



Chave e ponteiro gravados


In [5]:
chave_lida = int.from_bytes(papel_em_branco[7:11], byteorder='big')
print("Chave lida:", chave_lida)

ptr_lido = int.from_bytes(papel_em_branco[11:15], byteorder='big')
print("Ponteiro lido:", ptr_lido)

Chave lida: 42
Ponteiro lido: 100


### Node class simple test

In [6]:
from src.BplusTree import Node

In [7]:
pagina = bytearray(4096)

no = Node(pagina)

#definindo que é folha
no.setIsLeaf(True)
print("Esse nó é nó folha?", no.isLeaf())

#definindo quantas chaves tem no nó
no.setNumKeys(2)
print("Quantas chaves há nesse nó?", no.getNumKeys())

#definindo qual o valor das chaves no nó e seus ponteiros

no.setKey(0, 42)
no.setPointer(0, 100)
no.setKey(1, 10)
no.setPointer(2, 200)

print(f"O ponteiro {no.getPointer(0)} aponta para a chave {no.getKey(0)}")
print(f"O ponteiro {no.getPointer(1)} aponta para a chave {no.getKey(1)}")



Esse nó é nó folha? True
Quantas chaves há nesse nó? 2
O ponteiro 100 aponta para a chave 42
O ponteiro 0 aponta para a chave 10


## M4 - Tokenizer

In [8]:
from src.Tokenizer import Tokenizer

tk = Tokenizer()

sql1 = "SELECT * FROM aluno WHERE id = 42;"
print("teste1:")
for t in tk.tokenize(sql1):
    print(t)

sql2 = "CREATE TABLE aluno (id INT, nome CHAR(20));"
print("teste2:")
for t in tk.tokenize(sql2):
    print(t)

sql3 = "INSERT INTO aluno VALUES (1, 'Gabriela')"
print("teste3:")
for t in tk.tokenize(sql3):
    print(t)



teste1:
('KEYWORD', 'SELECT')
('SYMBOL', '*')
('KEYWORD', 'FROM')
('IDENTIFIER', 'aluno')
('KEYWORD', 'WHERE')
('IDENTIFIER', 'id')
('SYMBOL', '=')
('NUMBER', 42)
('SYMBOL', ';')
('EOF', None)
teste2:
('KEYWORD', 'CREATE')
('KEYWORD', 'TABLE')
('IDENTIFIER', 'aluno')
('SYMBOL', '(')
('IDENTIFIER', 'id')
('KEYWORD', 'INT')
('SYMBOL', ',')
('IDENTIFIER', 'nome')
('KEYWORD', 'CHAR')
('SYMBOL', '(')
('NUMBER', 20)
('SYMBOL', ')')
('SYMBOL', ')')
('SYMBOL', ';')
('EOF', None)
teste3:
('KEYWORD', 'INSERT')
('KEYWORD', 'INTO')
('IDENTIFIER', 'aluno')
('KEYWORD', 'VALUES')
('SYMBOL', '(')
('NUMBER', 1)
('SYMBOL', ',')
('STRING', 'Gabriela')
('SYMBOL', ')')
('EOF', None)


In [9]:
# #gera erro
# sql4 = "create table 123usuario (id INT)"
# print(tk.tokenize(sql4))

# M4 - Parser

In [10]:
from src.Parser import Parser

tk = Tokenizer()

# --- TESTE 1: CREATE TABLE ---
print("--- TESTE 1: CREATE ---")
sql_create = "CREATE TABLE aluno (id INT, nome CHAR(20));"
tokens_create = tk.tokenize(sql_create)
parser_create = Parser(tokens_create)
resultado_create = parser_create.parse()
print(resultado_create)
# Esperado: {'op': 'CREATE', 'table_name': 'aluno', 'columns': [{'name': 'id', 'type': 'INT', 'length': 4}, {'name': 'nome', 'type': 'CHAR', 'length': 20}]}

# --- TESTE 2: INSERT ---
print("\n--- TESTE 2: INSERT ---")
sql_insert = "INSERT INTO aluno VALUES (1, 'Gabriela');"
tokens_insert = tk.tokenize(sql_insert)
parser_insert = Parser(tokens_insert)
resultado_insert = parser_insert.parse()
print(resultado_insert)
# Esperado: {'op': 'INSERT', 'table_name': 'aluno', 'values': [1, 'Gabriela']}

# --- TESTE 3: SELECT com WHERE ---
print("\n--- TESTE 3: SELECT com WHERE ---")
sql_select_where = "SELECT * FROM aluno WHERE id = 42;"
tokens_select_where = tk.tokenize(sql_select_where)
parser_select_where = Parser(tokens_select_where)
resultado_select_where = parser_select_where.parse()
print(resultado_select_where)
# Esperado: {'op': 'SELECT', 'table_name': 'aluno', 'columns': ['*'], 'where': {'column': 'id', 'value': 42}}

# --- TESTE 4: SELECT sem WHERE ---
print("\n--- TESTE 4: SELECT sem WHERE ---")
sql_select_no_where = "SELECT id, nome FROM aluno;"
tokens_select_no_where = tk.tokenize(sql_select_no_where)
parser_select_no_where = Parser(tokens_select_no_where)
resultado_select_no_where = parser_select_no_where.parse()
print(resultado_select_no_where)
# Esperado: {'op': 'SELECT', 'table_name': 'aluno', 'columns': ['id', 'nome'], 'where': None}

# --- TESTE 5: Comando Inválido ---
print("\n--- TESTE 5: Erro de Sintaxe (Comando Inválido) ---")
sql_invalido = "UPDATE aluno SET id = 1;"
try:
    tokens_invalido = tk.tokenize(sql_invalido)
    parser_invalido = Parser(tokens_invalido)
    parser_invalido.parse()
except SyntaxError as e:
    print(f"Erro capturado com sucesso: {e}")
# Esperado: Erro capturado com sucesso: COMANDO DIGITADO INVÁLIDO. TENTE: CREATE, INSERT, SELECT

# --- TESTE 6: Lixo no final ---
print("\n--- TESTE 6: Erro de Sintaxe (Lixo no final) ---")
sql_lixo = "SELECT * FROM aluno lixo;"
try:
    tokens_lixo = tk.tokenize(sql_lixo)
    parser_lixo = Parser(tokens_lixo)
    parser_lixo.parse()
except SyntaxError as e:
    print(f"Erro capturado com sucesso: {e}")
# Esperado: Erro capturado com sucesso: TOKEN INESPERADO 'lixo' APÓS O FIM DO COMANDO

--- TESTE 1: CREATE ---
{'op': 'CREATE', 'table_name': 'aluno', 'columns': [{'name': 'id', 'type': 'INT', 'length': 4}, {'name': 'nome', 'type': 'CHAR', 'length': 20}]}

--- TESTE 2: INSERT ---
{'op': 'INSERT', 'table_name': 'aluno', 'values': [1, 'Gabriela']}

--- TESTE 3: SELECT com WHERE ---
{'op': 'SELECT', 'table_name': 'aluno', 'columns': ['*'], 'where': {'column': 'id', 'value': 42}}

--- TESTE 4: SELECT sem WHERE ---
{'op': 'SELECT', 'table_name': 'aluno', 'columns': ['id', 'nome'], 'where': None}

--- TESTE 5: Erro de Sintaxe (Comando Inválido) ---
Erro capturado com sucesso: COMANDO DIGITADO INVÁLIDO. TENTE: CREATE, INSERT, SELECT

--- TESTE 6: Erro de Sintaxe (Lixo no final) ---
Erro capturado com sucesso: TOKEN INESPERADO 'SELECT' APÓS O FIM DO COMANDO
